# 12a NSD one-session data, subj05

**Kaggle:** CPU, Internet on. **Inputs:** the output of 10a.

Data for the NSD one-session benchmark: subj05's 40 sessions of fsaverage single-trial betas, z-scored per session,
on the 39,161 vertices common to all 8 people; the first session's non-shared trials (training) and every repetition
of the 1,000 shared images (test). Checked against the person's Algonauts responses before saving. NSD's terms apply
to the output.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '12a-fewshotdata-subj05', 'cpu-build'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"beta_version": "betas_fithrf_GLMdenoise_RR", "sessions": 40, "trials_per_session": 750, "n_shared": 1000, "train_sessions": [1], "workers": 4, "min_check_r": 0.8, "min_check_images": 300}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']
TARGET = 'subj05'

import sys
import numpy as np
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.utils import config as C

import subprocess
import urllib.request
from concurrent.futures import ThreadPoolExecutor
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=True)
import nibabel as nib
import scipy.io as sio
TMP = Path("/kaggle/tmp/nsdbetas")
TMP.mkdir(parents=True, exist_ok=True)

def fetch(url, dst, tries=6):
    for i in range(tries):
        try:
            urllib.request.urlretrieve(url, dst)
            return dst
        except Exception as e:
            if i == tries - 1:
                raise
            print("  retry", url.split("/")[-1], type(e).__name__, flush=True)
            time.sleep(20 * (i + 1))

IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
common, cols = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)

d = sio.loadmat(fetch(f"{C.NSD_AWS}/nsddata/experiments/nsd/nsd_expdesign.mat", TMP / "design.mat"))
order = d["masterordering"].ravel().astype(int) - 1                      # trial -> index into the person's 10,000 images
p_idx = SUBJECTS.index(TARGET)
nsd_of = d["subjectim"][p_idx].astype(int) - 1                          # 0-based NSD (73k) ids, as 10a/10b
shared_ids = np.sort(d["sharedix"].ravel().astype(int) - 1)
if len(order) != FIXED["sessions"] * FIXED["trials_per_session"] or len(shared_ids) != FIXED["n_shared"]:
    raise ValueError("unexpected NSD design")
trial_ids = nsd_of[order]                                               # NSD id of every trial, in session order
is_shared = np.isin(trial_ids, shared_ids)
print(TARGET, "trials", len(trial_ids), "| shared-image trials", int(is_shared.sum()),
      "| shared images seen", len(np.unique(trial_ids[is_shared])), flush=True)

def session(sess):
    parts = []
    for h in ("lh", "rh"):
        f = fetch(f"{C.NSD_AWS}/nsddata_betas/ppdata/{TARGET}/fsaverage/{FIXED['beta_version']}/{h}.betas_session{sess:02d}.mgh",
                  TMP / f"{h}_{sess:02d}.mgh")
        b = np.asarray(nib.load(str(f)).dataobj, dtype=np.float32).reshape(-1, FIXED["trials_per_session"])
        parts.append(b[common[h]].T)
        f.unlink()
    y = np.concatenate(parts, 1)
    if y.shape != (FIXED["trials_per_session"], V) or not np.isfinite(y).all():
        raise ValueError(f"session {sess}: {y.shape}")
    return sess, ((y - y.mean(0)) / (y.std(0) + 1e-6)).astype(np.float16)  # z-score each vertex within the session

train_y, train_ids, test_y, test_trial_ids, test_session = [], [], [], [], []
with ThreadPoolExecutor(FIXED["workers"]) as pool:
    for sess, z in pool.map(session, range(1, FIXED["sessions"] + 1)):
        rows = np.arange((sess - 1) * FIXED["trials_per_session"], sess * FIXED["trials_per_session"])
        sh = is_shared[rows]
        if sess in FIXED["train_sessions"]:
            train_y.append(z[~sh])
            train_ids.append(trial_ids[rows][~sh])
        test_y.append(z[sh])
        test_trial_ids.append(trial_ids[rows][sh])
        test_session.append(np.full(int(sh.sum()), sess))
        print(f"session {sess:02d} | {(time.monotonic() - T0) / 60:.1f} min", flush=True)
train_y, train_ids = np.concatenate(train_y), np.concatenate(train_ids)
test_y, test_trial_ids, test_session = np.concatenate(test_y), np.concatenate(test_trial_ids), np.concatenate(test_session)
test_ids = np.unique(test_trial_ids)
reps = np.array([(test_trial_ids == i).sum() for i in test_ids])
test_mean = np.stack([test_y[test_trial_ids == i].astype(np.float32).mean(0) for i in test_ids])

# noise ceiling of the mean of each image's repetitions (NSD's formula on session-z-scored betas), as a correlation
full = reps == reps.max()
r3 = np.stack([test_y[test_trial_ids == i].astype(np.float32) for i in test_ids[full]])  # (images, reps, vertices)
noise = r3.var(1, ddof=1).mean(0)
snr2 = np.clip(1 - noise, 0, None) / np.maximum(noise, 1e-12)
ceiling = np.sqrt(snr2 / (snr2 + 1 / reps.max()))

# check against the person's own Algonauts responses (z-scored per session and averaged there): the same images must
# give the same patterns, which confirms vertex order, image ids and scaling before anything is saved
alg_ids = np.load(conv / f"{TARGET}_train_nsd_ids.npy")
alg = np.concatenate([np.load(conv / f"{TARGET}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][TARGET]] for h in ("lh", "rh")], 1)
row_of = {int(i): j for j, i in enumerate(alg_ids)}
both = [j for j, i in enumerate(test_ids) if int(i) in row_of]
a = np.asarray(alg[[row_of[int(test_ids[j])] for j in both]], np.float64)
b = test_mean[both].astype(np.float64)
pat_r = np.array([np.corrcoef(a[i], b[i])[0, 1] for i in range(len(both))])
print("check vs Algonauts:", len(both), "images | pattern r median", round(float(np.median(pat_r)), 3),
      "| 10th percentile", round(float(np.percentile(pat_r, 10)), 3), flush=True)
if len(both) < FIXED["min_check_images"] or np.median(pat_r) < FIXED["min_check_r"]:
    raise ValueError("12a's responses do not reproduce the Algonauts ones: vertex order, ids or scaling are wrong")

np.savez(C.PROJECT_ROOT / f"12a_{TARGET}.npz", train_y=train_y, train_ids=train_ids, test_ids=test_ids, test_mean=test_mean,
         test_reps=reps, test_y=test_y, test_trial_ids=test_trial_ids, test_session=test_session, ceiling=ceiling.astype(np.float32),
         shared_ids=shared_ids, common_lh=common["lh"], common_rh=common["rh"])
summary = {"subject": TARGET, "vertices": int(V), "train_trials": int(len(train_ids)), "train_images": int(len(np.unique(train_ids))),
           "test_images": int(len(test_ids)), "test_reps": {int(k): int(v) for k, v in zip(*np.unique(reps, return_counts=True))},
           "ceiling_median": float(np.median(ceiling)), "check_images": len(both), "check_pattern_r_median": float(np.median(pat_r)),
           "check_pattern_r_p10": float(np.percentile(pat_r, 10))}
(C.METRICS_DIR / f"12a_{TARGET}_summary.json").write_text(json.dumps(summary, indent=2))
print("12A_SUMMARY_JSON " + json.dumps(summary))